In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, f1_score, roc_auc_score
from skimage.feature import hog
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from joblib import Parallel, delayed



TRAIN_DIR = "data/agyikepek_4_osztaly/Training"
TEST_DIR = "data/agyikepek_4_osztaly/Testing"
IMG_SIZE = (128, 128)

CATEGORIES = ["glioma", "meningioma", "notumor", "pituitary"]


def preprocess_and_segment(image_path):
    """
    Hagyományos képfeldolgozás és alapfokú szegmentáció (háttér-maszkolás)
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # 1. Zajszűrés Gauss-szűrővel
    blurred = cv2.GaussianBlur(img, (5, 5), 0)
    
    # 2. SZEGMENTÁCIÓ: Otsu-féle automatikus küszöbölés a háttér elkülönítésére
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Maszk alkalmazása: csak az agyszövet marad meg, a külső zajok eltűnnek
    segmented_img = cv2.bitwise_and(img, img, mask=mask)
    
    # 3. Kontraszt növelés (CLAHE) a szegmentált régión
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_img = clahe.apply(segmented_img)
    
    # 4. Átméretezés a fix jellemzővektor mérethez
    resized_img = cv2.resize(enhanced_img, IMG_SIZE)
    
    # 5. JELLEMZŐKINYERÉS: HOG (Histogram of Oriented Gradients)
    features = hog(
        resized_img, 
        orientations=9, 
        pixels_per_cell=(16, 16), 
        cells_per_block=(2, 2), 
        visualize=False
    )
    
    # Alapvető statisztikai jellemzők hozzáadása (intenzitás eloszlás)
    mean_val = np.mean(resized_img)
    std_val = np.std(resized_img)
    final_features = np.append(features, [mean_val, std_val])
    
    return final_features

def load_dataset(base_dir):
    X, y = [], []
    for category_idx, category in enumerate(CATEGORIES):
        folder_path = os.path.join(base_dir, category)
        if not os.path.exists(folder_path):
            continue
        print(f" Feldolgozás: {category}...")
        for filename in os.listdir(folder_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(folder_path, filename)
                features = preprocess_and_segment(img_path)
                if features is not None:
                    X.append(features)
                    y.append(category_idx)
    return np.array(X), np.array(y)

# --- ADATOK BETÖLTÉSE GYORS ---

def process_single_image(filename, folder_path, category_idx):
    """Különálló függvény egyetlen kép feldolgozására (ezt futtatjuk majd a magokon)"""
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(folder_path, filename)
        features = preprocess_and_segment(img_path)
        if features is not None:
            return features, category_idx
    return None

def load_dataset_fast(base_dir):
    if not os.path.exists(base_dir):
        print(f"❌ HIBA: A fő mappa NEM LÉTEZIK: {base_dir}")
        return np.array([]), np.array([])
        
    # 1. Összegyűjtjük az összes feladatot egy listába
    tasks = []
    for category_idx, category in enumerate(CATEGORIES):
        folder_path = os.path.join(base_dir, category)
        if not os.path.exists(folder_path):
            continue
        for filename in os.listdir(folder_path):
            tasks.append((filename, folder_path, category_idx))
            
    print(f"⚡ Összegyűjtve: {len(tasks)} kép. Párhuzamos feldolgozás indítása az ÖSSZES CPU magon...")
    
    # 2. Lefuttatjuk párhuzamosan az összes elérhető CPU magon (n_jobs=-1)
    # A backend='loky' a legstabilabb dolog Windows alatt
    results = Parallel(n_jobs=-1, backend='loky')(
        delayed(process_single_image)(f, fp, idx) for f, fp, idx in tasks
    )
    
    # 3. Az eredmények szétválogatása
    X, y = [], []
    for res in results:
        if res is not None:
            X.append(res[0])
            y.append(res[1])
            
    return np.array(X), np.array(y)

In [ ]:
# --- ADATOK BETÖLTÉSE LASSÚ ---
print("--- 1. TANÍTÓ HALMAZ BETÖLTÉSE ---")
X_train, y_train = load_dataset(TRAIN_DIR)

print("\n--- 2. TESZTELŐ HALMAZ BETÖLTÉSE ---")
X_test, y_test = load_dataset(TEST_DIR)

In [ ]:
# --- ADATOK BETÖLTÉSE (GYORSÍTOTT VERZIÓ) ---
print("--- 1. TANÍTÓ HALMAZ BETÖLTÉSE ---")
X_train, y_train = load_dataset_fast(TRAIN_DIR)
print(f"✅ Sikeresen betöltve a tanító halmaz: {len(X_train)} kép.")

print("\n--- 2. TESZTELŐ HALMAZ BETÖLTÉSE ---")
X_test, y_test = load_dataset_fast(TEST_DIR)
print(f"✅ Sikeresen betöltve a tesztelő halmaz: {len(X_test)} kép.")

# Biztonsági ellenőrzés
if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError("❌ Valami hiba van, az adathalmaz üres maradt!")

In [ ]:

# --- MODELL COMPASS (TÖBB MODELL ÖSSZEHASONLÍTÁSA) ---
def get_fresh_models():
    return {
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "XGBoost": XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        "Support Vector Machine (SVM)": CalibratedClassifierCV(estimator=SVC(kernel='rbf', C=1.0, cache_size=2000, random_state=42), method='sigmoid', cv=3, ensemble=False),
        "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "Logistic Regression": LogisticRegression(max_iter=3000, C=1.0, random_state=42),
        "CalibratedClassifierCV (SVM kalibrált)": CalibratedClassifierCV(estimator=LinearSVC(C=1.0, dual=False, max_iter=3000, random_state=42), method='sigmoid', cv=3)
    }

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results_data = []

print("\n--- 3. MODELLEK TANÍTÁSA ÉS KIÉRTÉKELÉSE ---")
# DUPLA HUZAL: Végigmegyünk a Nem standardizált (False) és Standardizált (True) eseteken is
for use_scaling in [False, True]:
    condition = "Standardizált" if use_scaling else "Nem standardizált"
    print(f"\n==================================================")
    print(f"🔄 FUTTATÁS: {condition.upper()} ADATOKKAL")
    print(f"==================================================")
    
    # Adatok kiválasztása a kondíció alapján
    X_tr = X_train_scaled if use_scaling else X_train
    X_te = X_test_scaled if use_scaling else X_test
    
    current_models = get_fresh_models()
    
    for name, model in current_models.items():
        print(f"[{condition}] {name} tanítása folyamatban...")
        model.fit(X_tr, y_train)
        
        # Predikciók
        y_pred = model.predict(X_te)
        y_probs = model.predict_proba(X_te)
        
        # Metrikák
        acc = accuracy_score(y_test, y_pred)
        macro_f1 = f1_score(y_test, y_pred, average='macro')
        roc_auc = roc_auc_score(y_test, y_probs, multi_class='ovr', average='macro')
        
        # Eredmény mentése
        results_data.append({
            "Modell": name,
            "Előfeldolgozás": condition,
            "Accuracy": round(acc, 4),
            "Macro F1-Score": round(macro_f1, 4),
            "ROC-AUC (OvR)": round(roc_auc, 4)
        })
        print(f"-> Kész. Accuracy: {acc:.4f}")
        print(classification_report(y_test, y_pred, target_names=CATEGORIES))

# --- VÉGSŐ ÖSSZEHASONLÍTÓ TÁBLÁZAT ---
print("\n" + "="*60)
print("📊 VÉGSŐ ÖSSZEHASONLÍTÓ ÖSSZEGZÉS (ABÁCIÓS VIZSGÁLAT)")
print("="*60)
df_results = pd.DataFrame(results_data)

# Sorbarendezzük Modell név szerint, hogy egymás alatt legyen a skálázott és skálázatlan verziója
df_results = df_results.sort_values(by=["Modell", "Előfeldolgozás"], ascending=[True, False])
print(df_results.to_string(index=False))

print ("\n" + "="*60)
print("\n")

df_results2 = df_results.sort_values(by=["Accuracy"], ascending=[False])
print(df_results2.to_string(index=False))
